In [1]:
import os, re, time, subprocess, warnings
import numpy as np, pandas as pd, torch
from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification
warnings.filterwarnings('ignore')
OUT_DIR = '/kaggle/working'
DATA_DIR = '/kaggle/input/datasets/pavithra1mj/test-datset'
MODEL_ROOT = '/kaggle/input/notebooks/pavithra1mj/distillberta-model'
dev = 0 if torch.cuda.is_available() else -1

MODEL_DIR = next(r for r, d, fs in os.walk(MODEL_ROOT) if 'config.json' in fs and ('model.safetensors' in fs or 'pytorch_model.bin' in fs))
print(MODEL_DIR, os.listdir(MODEL_DIR))
vexts = ('.mp4', '.mkv', '.avi', '.mov', '.webm', '.wav', '.mp3', '.m4a')
search = DATA_DIR if os.path.exists(DATA_DIR) else '/kaggle/input'
files = [os.path.join(r, f) for r, _, fs in os.walk(search) for f in fs if f.lower().endswith(vexts)]
print(files)
VIDEO_PATH = files[0]
print(VIDEO_PATH, os.path.isfile(VIDEO_PATH))

/kaggle/input/notebooks/pavithra1mj/distillberta-model/distilbert_finetuned ['config.json', 'training_args.bin', 'tokenizer.json', 'tokenizer_config.json', 'model.safetensors']
['/kaggle/input/datasets/pavithra1mj/test-datset/WhatsApp Video 2026-10-03 at 11.15.52 AM.mp4']
/kaggle/input/datasets/pavithra1mj/test-datset/WhatsApp Video 2026-10-03 at 11.15.52 AM.mp4 True


In [2]:
t = time.time()
cls_tok = AutoTokenizer.from_pretrained(MODEL_DIR)
cls_model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR).to('cuda' if dev == 0 else 'cpu').eval()
clf = pipeline('text-classification', model=cls_model, tokenizer=cls_tok, device=dev, top_k=None)
asr = pipeline('automatic-speech-recognition', model='openai/whisper-base', device=dev, chunk_length_s=30, return_timestamps=True)
load_time = time.time() - t
print(load_time)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/1.98k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  290MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/245 [00:00<?, ?it/s]

...

added_tokens.json:   0%|          | 0.00/34.6k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.19k [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/185k [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'return_timestamps'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).


15.73521113395691


In [3]:
def run(video):
    T = {}
    t0 = time.time()
    wav = f'{OUT_DIR}/audio.wav'
    subprocess.run(['ffmpeg', '-y', '-i', video, '-vn', '-ac', '1', '-ar', '16000', wav], capture_output=True, check=True)
    T['audio_extraction_sec'] = time.time() - t0
    t1 = time.time()
    text = asr(wav)['text'].strip()
    T['transcription_sec'] = time.time() - t1
    sents = [s.strip() for s in re.split(r'(?<=[.!?])\s+', text) if s.strip()] or [text]
    t2 = time.time()
    preds = []
    for s in sents:
        r = max(clf(s)[0], key=lambda x: x['score'])
        preds.append({'sentence': s, 'label': r['label'], 'score': round(r['score'], 4)})
    T['classification_sec'] = time.time() - t2
    T['per_sentence_latency_ms'] = 1000 * T['classification_sec'] / max(1, len(sents))
    T['total_latency_sec'] = time.time() - t0
    return pd.DataFrame(preds), T

preds_df, T = run(VIDEO_PATH)
print(preds_df)

[transformers] Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
[transformers] Transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English. This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`. See https://github.com/huggingface/transformers/pull/28687 for more details.
[transformers] Passing `generation_config` together with generation-related arguments=({'suppress_tokens', 'begin_suppress_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer WhisperTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece token

                                             sentence   label   score
0   Good morning to all the respected teachers and...  NORMAL  0.9997
1   I'm happy to be a part of this retirement cere...  NORMAL  0.9997
2   I'm truly grateful and lucky to be a student M...  NORMAL  0.9998
3                   Your way of teaching was so good.  NORMAL  0.9998
4   You always gave clear and detailed explanation...  NORMAL  0.9998
5   And one because of you I was able to score a c...  NORMAL  0.9992
6   After attending the event too, she would appre...  NORMAL  0.9995
7   So thank you for being a wonderful teacher and...  NORMAL  0.9997
8           We will always cherish your guidance now.  NORMAL  0.9998
9   Once again, wishing you a very happy, healthy ...  NORMAL  0.9997
10                                         Thank you.  NORMAL  0.9997


In [4]:
preds_df, T = run(VIDEO_PATH)
dur = float(subprocess.run(['ffprobe', '-v', 'error', '-show_entries', 'format=duration', '-of', 'csv=p=0', VIDEO_PATH], capture_output=True, text=True).stdout.strip())
T['video_duration_sec'] = dur
T['real_time_factor'] = T['total_latency_sec'] / dur
T['num_sentences'] = len(preds_df)
T['model_load_sec'] = load_time
lat = pd.DataFrame([{k: round(v, 3) for k, v in T.items()}])
lat.to_csv(f'{OUT_DIR}/latency_distilbert.csv', index=False)
preds_df.to_csv(f'{OUT_DIR}/video_predictions_distilbert.csv', index=False)
print(lat.T)

                              0
audio_extraction_sec      0.502
transcription_sec         5.065
classification_sec        0.051
per_sentence_latency_ms   4.641
total_latency_sec         5.619
video_duration_sec       78.812
real_time_factor          0.071
num_sentences            11.000
model_load_sec           15.735


In [5]:
import base64
from IPython.display import HTML, display
for f in ['latency_distilbert.csv', 'video_predictions_distilbert.csv']:
    b = base64.b64encode(open(f'{OUT_DIR}/{f}', 'rb').read()).decode()
    display(HTML(f'<a download="{f}" href="data:application/octet-stream;base64,{b}">Download {f}</a>'))